# Django, Templates

## Introduction to Django Templates
---

In this lesson, you'll learn how to use **Django's template system** to generate HTML.

Templates are where your data meets the user interface - they combine HTML with dynamic content from your views.

1. [Template Syntax](#Template-Syntax),
    - [Variables](#Variables),
    - [Tags](#Tags),
    - [Filters](#Filters),
2. [Context and render()](#Context-and-render()),
    - [Passing Data to Templates](#Passing-Data-to-Templates),
    - [Context Processors](#Context-Processors),
3. [Template Inheritance](#Template-Inheritance),
    - [Base Templates](#Base-Templates),
    - [Blocks and Extends](#Blocks-and-Extends),
4. [Built-in Tags and Filters](#Built-in-Tags-and-Filters),
    - [Common Tags](#Common-Tags),
    - [Common Filters](#Common-Filters),
5. [🧠 EXERCISE 🧠, Create Templates for Bookstore](#🧠-EXERCISE-🧠,-Create-Templates-for-Bookstore).

<br>

## Template Syntax

---

Django templates use special syntax to mix HTML with dynamic content:

<br>

| Syntax | Purpose | Example |
|--------|---------|--------|
| `{{ }}` | Output variables | `{{ book.title }}` |
| `{% %}` | Template tags (logic) | `{% for book in books %}` |
| `{# #}` | Comments | `{# This is a comment #}` |

<br>

### Variables

---

Variables output values from the context. Use **dot notation** to access attributes, dictionary keys, list indices, or methods.

In [ ]:
# Template variable examples
<!-- Simple variable -->  # This is a oneline comment
<h1>{{ title }}</h1>

<!-- Object attribute -->
<p>{{ blog.title }}</p>

<!-- Dictionary key -->
<p>{{ user.name }}</p>

<!-- List index -->
<p>{{ items.0 }}</p>

<!-- Method call (no parentheses!) -->
<p>{{ name.upper }}</p>

In [ ]:
# View providing context for the template above

from django.shortcuts import get_object_or_404, render
from django.http import HttpRequest, HttpResponse
from models import Blog


def blog_detail_preview(request: HttpRequest, pk: int) -> HttpResponse:
    blog = get_object_or_404(Blog, pk=pk)
    return render(
        request,
        'blog/detail_preview.html',
        {'blog': blog},
    )

# curl "http://localhost:8000/blog/preview/1/"
# <!doctype html>
# <html>
#   <head>
#     <meta charset="utf-8">
#     <title>Blog Detail Preview</title>
#   </head>
#   <body>
#     <h1>Blog Detail Preview</h1>
    
#       <p><strong>ID:</strong> 1</p>
#       <p><strong>Title:</strong> Django Bootcamp</p>
#       <p><strong>Author:</strong> Matous Holinka</p>
#       <p><strong>Published:</strong> 2026-03-26</p>
    
#   </body>
# </html>

```html
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Blog Detail Preview</title>
  </head>
  <body>
    <h1>Blog Detail Preview</h1>
    {% if blog %}
      <p><strong>ID:</strong> {{ blog.id }}</p>
      <p><strong>Title:</strong> {{ blog.title }}</p>
      <p><strong>Author:</strong> {{ blog.author }}</p>
      <p><strong>Published:</strong> {{ blog.published_date|date:"Y-m-d" }}</p>
    {% else %}
      <p>No blog found.</p>
    {% endif %}
  </body>
</html>
```

<br>

**Variable resolution order:**

When you write `{{ foo.bar }}`, Django tries in this order:

1. Dictionary lookup: `foo["bar"]`
2. Attribute lookup: `foo.bar`
3. List index: `foo[bar]` (if bar is numeric)

<br>

### Tags

---

Tags provide **logic** in templates - loops, conditionals, includes, etc.

In [ ]:
# Common template tags

template_tags = """
<!-- For loop -->
{% for book in books %}
    <p>{{ book.title }}</p>
{% endfor %}

<!-- For loop with empty, if there is an empty value, then do whatever is inside the empty block -->
{% for book in books %}
    <p>{{ book.title }}</p>
{% empty %}
    <p>No books available.</p>
{% endfor %}

<!-- If/elif/else -->
{% if user.is_authenticated %}
    <p>Welcome, {{ user.username }}!</p>
{% elif user.is_guest %}
    <p>Welcome, guest!</p>
{% else %}
    <p>Please log in.</p>
{% endif %}

<!-- URL tag -->
<a href="{% url 'catalog:book_detail' pk=book.pk %}">View Book</a>
"""
print(template_tags)

<br>

### Filters

---

Filters **transform** variable values. Use the pipe `|` symbol.

In [ ]:
# Template filter examples

template_filters = """
<!-- Lowercase -->
{{ title|lower }}

<!-- Uppercase -->
{{ title|upper }}

<!-- Title case -->
{{ name|title }}

<!-- Default value -->
{{ description|default:"No description" }}

<!-- Truncate -->
{{ long_text|truncatewords:20 }}

<!-- Date formatting -->
{{ published_date|date:"F j, Y" }}

<!-- Length -->
{{ items|length }} items

<!-- Chaining filters -->
{{ name|lower|truncatewords:5 }}
"""
print(template_filters)

The examples above are built-in. If you know the name of the **filter**, you can use them immediately.

<br>

But you can even define your own custom filters.

In [ ]:
# blog/templatetags/blog_filters.py
from django import template

register = template.Library()


@register.filter
def upper_surname(text):
    """Get the upper surname from the text."""
    try:
        first_name, last_name = text.split(' ')
        return ' '.join([first_name, last_name.upper()])
    
    except IndexError:
        return text.upper()

Write the surname in uppercase and the first name in lowercase.

Don't forget:
1. Include or load the filter to your template,
2. Restart the server.

```html
{% load blog_filters %}
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Blog Detail Preview</title>
  </head>
  <body>
    <h1>Blog Detail Preview</h1>
    {% if blog %}
      <p><strong>ID:</strong> {{ blog.id }}</p>
      <p><strong>Title:</strong> {{ blog.title }}</p>
      <p><strong>Author:</strong> {{ blog.author|upper_surname }}</p>
      <p><strong>Published:</strong> {{ blog.published_date|date:"Y-m-d" }}</p>
    {% else %}
      <p>No blog found.</p>
    {% endif %}
  </body>
</html>
```

<br>

## Context and render()

---

### Passing Data to Templates

---

The **context** is a dictionary of data passed from the view to the template.

In [ ]:
# Example: Passing context to template

from django.shortcuts import get_object_or_404, render
from django.http import HttpRequest, HttpResponse
from blog.models import Blog


def blog_detail_preview(request: HttpRequest, pk: int) -> HttpResponse:
    blog = get_object_or_404(Blog, pk=pk)

    # Context keys become template variable names
    context = {
        'blog': blog,
        'page_title': 'Blog Detail Preview',
        'show_published_date': True,
        'author_name': blog.author,
        'blog_name': blog.title,
    }

    return render(request, 'blog/detail_preview.html', context)

```html
{% load blog_filters %}
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Blog Detail Preview</title>
  </head>
  <body>
    <h1>Blog Detail Preview</h1>
    {% if blog %}
      <p><strong>ID:</strong> {{ blog.id }}</p>
      <p><strong>Title:</strong> {{ blog.title }}</p>
      <p><strong>Author:</strong> {{ blog.author|upper_surname }}</p>
      <p><strong>Published:</strong> {{ blog.published_date|date:"Y-m-d" }}</p>
      <p><strong>Page Title:</strong> {{ page_title }}</p>
    {% else %}
      <p>No blog found.</p>
    {% endif %}
  </body>
</html>

```

Yes, you can pass the `context` dictionary directly into the `render()` function.

**Direct approach:** Best for simple views with only one or two variables. It keeps the code short.

**Variable approach** (`context = {}`): Better for complex views. It makes the code much more readable when you're passing a lot of data or performing logic before rendering the page."

<br>

### Context Processors

---

**Context processors** automatically add variables to every template context.

Django includes several by default:

| Processor | Provides |
|-----------|----------|
| `request` | The `request` object |
| `auth` | `user` and `perms` objects |
| `messages` | Flash messages |
| `debug` | Debug info (when DEBUG=True) |

In [ ]:
# Using built-in context variables in templates

template_context_vars = """
<!-- User from auth context processor -->
{% if user.is_authenticated %}
    <p>Logged in as: {{ user.username }}</p>
{% else %}
    <a href="{% url 'login' %}">Log in</a>
{% endif %}

<!-- Request from request context processor -->
<p>Current path: {{ request.path }}</p>

<!-- Messages from messages context processor -->
{% for message in messages %}
    <div class="alert alert-{{ message.tags }}">
        {{ message }}
    </div>
{% endfor %}
"""
print(template_context_vars)

<br>

## Template Inheritance

---

### Base Templates

---

Template inheritance lets you define a **base template** with common HTML structure, then **extend** it in child templates.

This follows the **DRY principle** (Don't Repeat Yourself).

#### Base template: `templates/blog/base.html`
```html
<!DOCTYPE html>
<html>
<head>
    <title>{% block title %}Blog{% endblock %}</title>
</head>
<body>
    <nav>
        <a href="/blog/">All Blogs</a> |
        <a href="/blog/about/">About</a>
    </nav>

    <main>
        {% block content %}{% endblock %}
    </main>

    <footer>
        <p>My Blog</p>
    </footer>
</body>
</html>
```

<br>

### Blocks and Extends

---

Child templates use `{% extends %}` to inherit from a base template and `{% block %}` to override sections.

#### Another child template: `templates/blog/detail_preview.html`

```html
{% extends "blog/base.html" %}
{% load blog_filters %}
<!DOCTYPE html>

{% block title %}{{ block.super }} | Blog Detail{% endblock %}

{% block content %}
  <h1>Blog Detail Preview</h1>
  {% if blog %}
    <p><strong>ID:</strong> {{ blog.id }}</p>
    <p><strong>Title:</strong> {{ blog.title }}</p>
    <p><strong>Author:</strong> {{ blog.author|upper_surname }}</p>
    <p><strong>Published:</strong> {{ blog.published_date|date:"Y-m-d" }}</p>
    {% if page_title %}
      <p><strong>Page Title:</strong> {{ page_title }}</p>
    {% endif %}
  {% else %}
    <p>No blog found.</p>
  {% endif %}
{% endblock %}
```

```bash
curl "http://localhost:8000/blog/preview/1/"
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>My Blog | Blog Detail</title>
    
  </head>
  <body>
    <nav>
      <a href="/blog/">All Blogs</a> |
      <a href="/blog/search/">Search</a> |
      <a href="/blog/preview/1/">Preview</a>
    </nav>

    <main>
      
  <h1>Blog Detail Preview</h1>
  
    <p><strong>ID:</strong> 1</p>
    <p><strong>Title:</strong> Django Bootcamp</p>
    <p><strong>Author:</strong> Matous HOLINKA</p>
    <p><strong>Published:</strong> 2026-03-26</p>
    
      <p><strong>Page Title:</strong> Blog Detail Preview</p>
    
  

    </main>

    <footer>
      <p>My Blog</p>
    </footer>
  </body>
</html>
```

<br>

**Using `{{ block.super }}`** to include parent content:

In [ ]:
# Example: Extending parent block content

block_super_example = """
{% extends "blog/base.html" %}
{% load blog_filters %}

{% load static %}

{% block title %}{{ block.super }} | Blog List{% endblock %}
{# Result: "My Blog | Blog List" #}

{% block extra_css %}
{{ block.super }}
<link rel="stylesheet" href="{% static 'css/example_list.css' %}">
{% endblock %}

{% block content %}
    <h1>Available Blogs</h1>
    <ul>
        <li>{{ blogs.0.title|garbled_text }}</li>
    </ul>
{% endblock %}
"""
print(block_super_example)

In Django templates, `{{ block.super }}` is used for Template Inheritance.

By default, when you override a `{% block %}` in a child template, it replaces everything from the parent.

<br>

## Built-in Tags and Filters

---

### Common Tags

---

Tags control the logic of the template.

They tell the template what to do (e.g. repeat text in a loop, evaluate a condition, or load another file).

In [ ]:
# if/else tag
{% extends "blog/base.html" %}  # Also built-in tag
{% load blog_filters %}

{% block title %}{{ block.super }} | Blog Detail{% endblock %}

{% block content %}
  <h1>Blog Detail Preview</h1>
  {% if blog %}
    <p><strong>ID:</strong> {{ blog.id }}</p>
    <p><strong>Title:</strong> {{ blog.title }}</p>
    <p><strong>Author:</strong> {{ blog.author|upper_surname }}</p>
    <p><strong>Published:</strong> {{ blog.published_date|date:"Y-m-d" }}</p>
    {% if page_title %}
      <p><strong>Page Title:</strong> {{ page_title }}</p>
    {% endif %}
  {% else %}
    <p>No blog found.</p>
  {% endif %}
{% endblock %}


The `{% url %}` tag generates a URL based on the name in `urls.py` — instead of having to type the URL out loud.
```bash
<!-- WRONG -->
<a href="/blog/1/">Detail</a>

<!-- CORRECT -->
<a href="{% url 'blog:blog_detail' pk=1 %}">Detail</a>
```

In [ ]:
# static tag - for static files
{% load static %}

<link rel="stylesheet" href="{% static 'css/style.css' %}">
<script src="{% static 'js/app.js' %}"></script>
<img src="{% static 'images/logo.png' %}" alt="Logo">

<br>

### Common Filters

---

Filters are used to modify and format variables. They do not solve the logic "around" it, but they change how a particular value is displayed to the user.

In [ ]:
# String filters

string_filters = """
{{ name|lower }}                {# lowercase #}
{{ name|upper }}                {# UPPERCASE #}
{{ name|title }}                {# Title Case #}
{{ name|capfirst }}             {# First letter capitalized #}
{{ text|truncatewords:20 }}     {# Truncate to 20 words #}
{{ text|truncatechars:100 }}    {# Truncate to 100 characters #}
{{ text|linebreaks }}           {# Convert newlines to <p> and <br> #}
{{ text|linebreaksbr }}         {# Convert newlines to <br> #}
{{ text|striptags }}            {# Remove HTML tags #}
{{ text|slugify }}              {# URL-safe slug #}
"""
print(string_filters)

In [ ]:
# Date and number filters

date_number_filters = """
<!-- Date formatting -->
{{ published|date:"F j, Y" }}        {# January 15, 2024 #}
{{ published|date:"Y-m-d" }}         {# 2024-01-15 #}
{{ published|date:"SHORT_DATE_FORMAT" }}
{{ published|timesince }}            {# 3 days ago #}

<!-- Number formatting -->
{{ price|floatformat:2 }}            {# 29.99 #}
{{ count|add:10 }}                   {# Add 10 to count #}
{{ items|length }}                   {# List length #}
{{ value|divisibleby:3 }}            {# True if divisible by 3 #}
"""
print(date_number_filters)

If you need [official docs](https://docs.djangoproject.com/en/6.0/ref/templates/builtins/#date)

In [ ]:
# Default and conditional filters

default_filters = """
<!-- Default values -->
{{ description|default:"No description" }}
{{ value|default_if_none:"N/A" }}

<!-- Boolean displays -->
{{ is_active|yesno:"Active,Inactive,Unknown" }}

<!-- Pluralize -->
{{ count }} book{{ count|pluralize }}
{{ count }} categor{{ count|pluralize:"y,ies" }}
"""
print(default_filters)

In [ ]:
# List filters
list_filters = """
{{ items|first }}                {# First item #}
{{ items|last }}                 {# Last item #}
{{ items|length }}               {# Number of items #}
{{ items|join:", " }}            {# Join with separator #}
{{ items|slice:":5" }}           {# First 5 items #}
{{ items|random }}               {# Random item #}
"""
print(list_filters)

<br>

## Template Organization

---

**Recommended directory structure:**

```
blog/
├── templates/                  # Project-wide templates
│   ├── base.html               # Base template
│   ├── about.html        
│   ├── blog_list_static_example.html              
│   ├── detail_preview.html              
│   ├── partials/               # Reusable components
│   │   ├── header.html
│   │   ├── footer.html
│   │   └── blog_card.html
│   └── registration/           # Auth templates
│       ├── login.html
│       └── logout.html
...
```

<br>

**Configure in settings.py:**

In [ ]:
# settings.py template configuration

from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

TEMPLATES = [
    {
        'BACKEND': 'django.template.backends.django.DjangoTemplates',
        'DIRS': [
            BASE_DIR / 'templates',  # Project-wide templates
        ],
        'APP_DIRS': True,  # Look in app/templates/ directories
        'OPTIONS': {
            'context_processors': [
                'django.template.context_processors.debug',
                'django.template.context_processors.request',
                'django.contrib.auth.context_processors.auth',
                'django.contrib.messages.context_processors.messages',
            ],
        },
    },
]

`BACKEND` — Tells Django that we are using its own built-in template engine.

`DIRS` — A list of folders where Django looks for templates globally (outside of specific apps). `BASE_DIR / 'templates'` points to a templates/ folder in your project's root directory.

`APP_DIRS: True` — Django automatically searches for templates inside the <app>/templates/ folder of every installed app. This is why our blog/templates/blog/ structure works out of the box.

`context_processors` — These are functions that automatically inject specific variables into every single template

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Variables** | `{{ variable }}` - output values |
| **Tags** | `{% tag %}` - logic (for, if, url, etc.) |
| **Filters** | `{{ value\|filter }}` - transform values |
| **Context** | Dictionary passed from view to template |
| **render()** | `render(request, 'template.html', context)` |
| **Inheritance** | `{% extends 'base.html' %}` + `{% block %}` |
| **Include** | `{% include 'partial.html' %}` for reuse |
| **Static** | `{% load static %}` + `{% static 'file.css' %}` |

<br>

### 🧠 EXERCISE 🧠, Create Templates for Bookstore

---

Create a small template exercise for the current blog project:

1. Create/verify `blog/templates/blog/base.html` with:
   - Basic HTML structure
   - Simple navigation (`/blog/`, `/blog/search/`, `/blog/preview/1/`)
   - Blocks: `title`, `content` (optional: `extra_css`)
2. Create/verify `blog/templates/blog/detail_preview.html` that:
   - Extends `blog/base.html`
   - Shows blog `id`, `title`, `author`, `published_date`
   - Shows fallback text when `blog` is missing
3. Ensure `blog_detail_preview` view uses:
   - `render(request, 'blog/detail_preview.html', {'blog': blog})`
4. Test in browser:
   - `/blog/preview/1/`


<details>
    <summary>▶️ Solution</summary>

**Validation (why old version was off):**
- It referenced `{% url 'blog:about' %}`, but this route is not in current `blog/urls.py`.
- It used `blog_list`/`blog_detail` template flow, while current project focuses on `blog_detail_preview`.
- It asked for 3 templates and larger refactor; current lesson context is better served by 1-2 short templates.

**1. `blog/templates/blog/base.html`:**

```html
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>{% block title %}My Blog{% endblock %}</title>
    {% block extra_css %}{% endblock %}
  </head>
  <body>
    <nav>
      <a href="/blog/">All Blogs</a> |
      <a href="/blog/search/">Search</a> |
      <a href="/blog/preview/1/">Preview</a>
    </nav>

    <main>
      {% block content %}{% endblock %}
    </main>
  </body>
</html>
```

**2. `blog/templates/blog/detail_preview.html`:**

```html
{% extends "blog/base.html" %}
{% load blog_filters %}

{% block title %}{{ block.super }} | Blog Detail{% endblock %}

{% block content %}
  <h1>Blog Detail Preview</h1>
  {% if blog %}
    <p><strong>ID:</strong> {{ blog.id }}</p>
    <p><strong>Title:</strong> {{ blog.title }}</p>
    <p><strong>Author:</strong> {{ blog.author|upper_surname }}</p>
    <p><strong>Published:</strong> {{ blog.published_date|date:"Y-m-d" }}</p>
  {% else %}
    <p>No blog found.</p>
  {% endif %}
{% endblock %}
```

**3. View snippet (`blog/views.py`):**

```python
def blog_detail_preview(request: HttpRequest, pk: int) -> HttpResponse:
    blog = get_object_or_404(Blog, pk=pk)
    return render(request, 'blog/detail_preview.html', {'blog': blog})
```

**4. Test:**

```bash
python manage.py runserver
curl http://127.0.0.1:8000/blog/preview/1/
```
</details>


<br>

### 🧠 EXERCISE 🧠, Add Template Filters and Conditions

---

Enhance the blog list template:

1. Show blog count: "Showing X blog(s)"
2. Display `published_date` formatted as "Month Day, Year"
3. Highlight blogs published after 2023 with a "New" label
4. Add a link to each blog detail page
5. Show "No blogs available." when list is empty

<details>
    <summary>▶️ Solution</summary>

**1. Update `blog/views.py`:**

```python
from django.shortcuts import render
from .models import Blog

def blog_list(request):
    blogs = Blog.objects.all()
    return render(request, 'blog/blog_list.html', {'blogs': blogs})
```

**2-5. Update `blog/templates/blog/blog_list.html`:**

```html
{% extends "blog/base.html" %}

{% block title %}Blogs - My Blog{% endblock %}

{% block content %}
<h1>All Blogs</h1>
<p>Showing {{ blogs|length }} blog{{ blogs|length|pluralize }}.</p>

{% for blog in blogs %}
    <div style="margin-bottom: 20px; padding: 10px; border: 1px solid #ccc;">
        <h2>
            {{ blog.title }}
            {% if blog.published_date.year > 2023 %}
                <span style="color: green;">🆕 New</span>
            {% endif %}
        </h2>
        <p>Author: {{ blog.author }}</p>
        <p>Published: {{ blog.published_date|date:"F j, Y" }}</p>
        <a href="{% url 'blog:blog_detail' pk=blog.id %}">Read more</a>
    </div>
{% empty %}
    <p>No blogs available.</p>
{% endfor %}
{% endblock %}
```
</details>

---